# Export patch products from `.npz` tensors

Post-process tensors from `dem_tensor_pipeline.py` and export **three products per patch** using **only** the `.npz` files (no external DEM dirs):

| # | Product | Filename |
|---|---------|----------|
| 1 | LiDAR photon points | `{n}_LiDAR_{pXXXXX}.shp` |
| 2 | 10 m bicubic FABDEM | `{n}_FABDEM_10mbicubic_{pXXXXX}.tif` |
| 3 | 30 m FABDEM (from Ch1) | `{n}_FABDEM_30m_{pXXXXX}.tif` |

**30 m note:** Ch1 is already 10 m bicubic. Product 3 is a **3× block-mean downsample** of Ch1 (not native 30 m FABDEM).

**Deps:** `numpy`, `rasterio`, `fiona`, `shapely`
```
pip install numpy rasterio fiona shapely
```


## 1. Imports

In [1]:
!pip install numpy rasterio fiona shapely

In [2]:
import os
import re
import gc
import traceback

import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.crs import CRS

import fiona
from fiona.crs import from_epsg as fiona_from_epsg
from shapely.geometry import Point, mapping

print("imports OK")


imports OK


## 2. Paths & config

Edit these to match your machine. Outputs are created under `OUT_BASE`.


In [8]:
# Inputs — tensors only
NPZ_DIR = r"D:\Vaibhav\dem-lidar\temp"

# Outputs (new folders next to the tensor set)
OUT_BASE = r"D:\Vaibhav\dem-lidar\tensors_256\train"
OUT_LIDAR_SHP = r"D:\Vaibhav\dem-lidar\inference\lidar_shp"
OUT_FABDEM_10M = r"D:\Vaibhav\dem-lidar\inference\fabdem_10m_bicubic"
OUT_FABDEM_30M = r"D:\Vaibhav\dem-lidar\inference\fabdem_30m"

# 10 m → 30 m downsample factor (pixel size × 3)
DOWNSAMPLE_FACTOR = 3

print("NPZ_DIR       :", NPZ_DIR)
print("OUT_LIDAR_SHP :", OUT_LIDAR_SHP)
print("OUT_FABDEM_10M:", OUT_FABDEM_10M)
print("OUT_FABDEM_30M:", OUT_FABDEM_30M)
print("DOWNSAMPLE_FACTOR:", DOWNSAMPLE_FACTOR)


NPZ_DIR       : D:\Vaibhav\dem-lidar\temp
OUT_LIDAR_SHP : D:\Vaibhav\dem-lidar\inference\lidar_shp
OUT_FABDEM_10M: D:\Vaibhav\dem-lidar\inference\fabdem_10m_bicubic
OUT_FABDEM_30M: D:\Vaibhav\dem-lidar\inference\fabdem_30m
DOWNSAMPLE_FACTOR: 3


## 3. Helper functions

In [9]:
# basename from pipeline: {photon:05d}_{tile_stem}_p{idx:05d}.npz
_NPZ_RE = re.compile(
    r"^(?P<photons>\d+)_(?P<tile>.+)_p(?P<serial>\d+)\.npz$",
    re.IGNORECASE,
)


def parse_npz_name(filename):
    """Return (photon_count_int, serial_str like 'p00042', tile_stem) or None."""
    m = _NPZ_RE.match(filename)
    if not m:
        return None
    photons = int(m.group("photons"))
    serial = f"p{int(m.group('serial')):05d}"
    tile = m.group("tile")
    return photons, serial, tile


def product_basename(photons, kind, serial):
    """
    kind ∈ {'LiDAR', 'FABDEM_10mbicubic', 'FABDEM_30m'}
    →  '{photons}_{kind}_{serial}'
    """
    return f"{photons}_{kind}_{serial}"


def gdal_gt_to_affine(gt6):
    """GDAL (x0, dx, rx, y0, ry, dy) → rasterio Affine."""
    x0, dx, rx, y0, ry, dy = [float(v) for v in gt6]
    return Affine(dx, rx, x0, ry, dy, y0)


def pixel_centers_xy(rows, cols, gt6):
    """
    Map pixel (row, col) → map coords of pixel *center* using GDAL geotransform.
      x = x0 + (col+0.5)*dx + (row+0.5)*rx
      y = y0 + (col+0.5)*ry + (row+0.5)*dy
    """
    x0, dx, rx, y0, ry, dy = [float(v) for v in gt6]
    c = cols.astype(np.float64) + 0.5
    r = rows.astype(np.float64) + 0.5
    xs = x0 + c * dx + r * rx
    ys = y0 + c * ry + r * dy
    return xs, ys


def write_geotiff(path, array2d, transform, crs, nodata=None):
    h, w = array2d.shape
    bx = max(16, (min(256, w) // 16) * 16) or 16
    by = max(16, (min(256, h) // 16) * 16) or 16
    tiled = (h >= by) and (w >= bx)
    profile = {
        "driver": "GTiff",
        "height": h,
        "width": w,
        "count": 1,
        "dtype": "float32",
        "crs": crs,
        "transform": transform,
        "compress": "deflate",
        "tiled": tiled,
    }
    if tiled:
        profile["blockxsize"] = bx
        profile["blockysize"] = by
    if nodata is not None:
        profile["nodata"] = float(nodata)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(np.asarray(array2d, dtype=np.float32), 1)


def downsample_fabdem_to_30m(fab10, transform10, factor=DOWNSAMPLE_FACTOR):
    """
    Derive a ~30 m FABDEM from 10 m Ch1 only.

    - Block-mean over non-overlapping factor×factor windows.
    - Truncates remainder rows/cols that do not fill a full block.
    - Scales the Affine so pixel size becomes 10m * factor (typically 30 m)
      while keeping the same top-left corner.

    Returns (array_30m, transform_30m).
    """
    h, w = fab10.shape
    h30 = h // factor
    w30 = w // factor
    if h30 < 1 or w30 < 1:
        raise ValueError(f"Patch too small to downsample by {factor}: got {h}x{w}")

    cropped = fab10[: h30 * factor, : w30 * factor].astype(np.float64)
    arr30 = (
        cropped.reshape(h30, factor, w30, factor)
        .mean(axis=(1, 3))
        .astype(np.float32)
    )

    t = transform10
    transform30 = Affine(
        t.a * factor,
        t.b * factor,
        t.c,
        t.d * factor,
        t.e * factor,
        t.f,
    )
    return arr30, transform30


def write_lidar_shapefile_full(path_no_ext, xs, ys, zs, rows, cols, epsg):
    """
    Write ESRI Shapefile of photon points.
    Attributes: height_m, pixel_row, pixel_col. CRS = patch EPSG.
    path_no_ext: full path without .shp extension.
    """
    schema = {
        "geometry": "Point",
        "properties": {
            "height_m": "float",
            "pixel_row": "int",
            "pixel_col": "int",
        },
    }
    try:
        crs = fiona_from_epsg(int(epsg)) if epsg and int(epsg) > 0 else None
    except Exception:
        crs = None

    os.makedirs(os.path.dirname(path_no_ext) or ".", exist_ok=True)
    with fiona.open(
        path_no_ext + ".shp",
        "w",
        driver="ESRI Shapefile",
        crs=crs,
        schema=schema,
    ) as sink:
        for x, y, z, r, c in zip(xs, ys, zs, rows, cols):
            sink.write(
                {
                    "geometry": mapping(Point(float(x), float(y))),
                    "properties": {
                        "height_m": float(z),
                        "pixel_row": int(r),
                        "pixel_col": int(c),
                    },
                }
            )

    prj_path = path_no_ext + ".prj"
    if (crs is None) and epsg and int(epsg) > 0 and not os.path.exists(prj_path):
        try:
            wkt = CRS.from_epsg(int(epsg)).to_wkt()
            with open(prj_path, "w", encoding="utf-8") as f:
                f.write(wkt)
        except Exception:
            pass


print("helpers ready")


helpers ready


## 4. Run export loop

Creates output folders, walks every `.npz` in `NPZ_DIR`, and writes the three products.


In [ ]:
for d in (OUT_LIDAR_SHP, OUT_FABDEM_10M, OUT_FABDEM_30M):
    os.makedirs(d, exist_ok=True)

if not os.path.isdir(NPZ_DIR):
    raise FileNotFoundError(f"NPZ_DIR not found: {NPZ_DIR}")

npz_files = sorted(f for f in os.listdir(NPZ_DIR) if f.lower().endswith(".npz"))
print(f"Found {len(npz_files)} tensor files in {NPZ_DIR}")
print("Mode: all products from .npz only (30 m = 3× block-mean of Ch1)")

stats = {
    "ok": 0,
    "bad_name": 0,
    "lidar": 0,
    "fab10": 0,
    "fab30": 0,
    "empty_lidar": 0,
    "errors": 0,
}

for i, fname in enumerate(npz_files):
    parsed = parse_npz_name(fname)
    if not parsed:
        print(f"   ⚠️ skip (name pattern): {fname}")
        stats["bad_name"] += 1
        continue

    photons_name, serial, tile_stem = parsed
    npz_path = os.path.join(NPZ_DIR, fname)

    try:
        with np.load(npz_path) as z:
            tensor = z["tensor"]  # (4, H, W)
            gt6 = np.asarray(z["geotransform"], dtype=np.float64)
            epsg = int(z["epsg"])

        ch1_fab10 = tensor[0]
        ch2_truth = tensor[1]
        ch3_mask = tensor[2]
        H, W = ch1_fab10.shape
        transform = gdal_gt_to_affine(gt6)
        crs = CRS.from_epsg(epsg) if epsg > 0 else None

        rows_idx, cols_idx = np.where(ch3_mask > 0.5)
        n_photons = int(len(rows_idx))
        photons = n_photons if n_photons > 0 else photons_name

        print(
            f"\n[{i + 1}/{len(npz_files)}] {fname} | "
            f"size {H}x{W} | photons {photons} | EPSG:{epsg} | {serial}"
        )

        # 1) LiDAR shapefile
        lidar_base = product_basename(photons, "LiDAR", serial)
        lidar_path = os.path.join(OUT_LIDAR_SHP, lidar_base)

        if n_photons == 0:
            print("   ⚠️ no photons in mask — writing empty shapefile schema only")
            stats["empty_lidar"] += 1
            write_lidar_shapefile_full(
                lidar_path,
                np.array([]), np.array([]), np.array([]),
                np.array([]), np.array([]),
                epsg,
            )
        else:
            zs = ch2_truth[rows_idx, cols_idx]
            xs, ys = pixel_centers_xy(rows_idx, cols_idx, gt6)
            write_lidar_shapefile_full(
                lidar_path, xs, ys, zs, rows_idx, cols_idx, epsg
            )
            print(f"   ✅ LiDAR  → {lidar_base}.shp  ({n_photons} points)")
        stats["lidar"] += 1

        # 2) 10 m bicubic FABDEM (Ch1 as-is)
        fab10_base = product_basename(photons, "FABDEM_10mbicubic", serial)
        fab10_path = os.path.join(OUT_FABDEM_10M, fab10_base + ".tif")
        write_geotiff(fab10_path, ch1_fab10, transform, crs)
        print(f"   ✅ FAB10  → {fab10_base}.tif  ({H}x{W})")
        stats["fab10"] += 1

        # 3) 30 m FABDEM — downsample Ch1 only
        fab30_base = product_basename(photons, "FABDEM_30m", serial)
        fab30_path = os.path.join(OUT_FABDEM_30M, fab30_base + ".tif")
        arr30, tfm30 = downsample_fabdem_to_30m(
            ch1_fab10, transform, factor=DOWNSAMPLE_FACTOR
        )
        write_geotiff(fab30_path, arr30, tfm30, crs)
        dx30 = abs(tfm30.a)
        print(
            f"   ✅ FAB30  → {fab30_base}.tif  "
            f"({arr30.shape[0]}x{arr30.shape[1]} @ ~{dx30:.1f} m) "
            f"[block-mean ×{DOWNSAMPLE_FACTOR} from Ch1]"
        )
        stats["fab30"] += 1

        stats["ok"] += 1

    except Exception as exc:
        stats["errors"] += 1
        print(f"   ❌ ERROR on {fname}: {exc}")
        traceback.print_exc()
        continue

    gc.collect()

print("\n==================================================")
print("         EXPORT PATCH PRODUCTS COMPLETE          ")
print("==================================================")
print(f"✅ Patches processed:     {stats['ok']:,}")
print(f"✅ LiDAR shapefiles:      {stats['lidar']:,}  (empty: {stats['empty_lidar']:,})")
print(f"✅ FABDEM 10 m TIFFs:     {stats['fab10']:,}")
print(f"✅ FABDEM 30 m TIFFs:     {stats['fab30']:,}  (from Ch1 downsample)")
print(f"❌ Bad names:             {stats['bad_name']:,}")
print(f"❌ Errors:                {stats['errors']:,}")
print("-" * 50)
print(f"📁 LiDAR SHP:     {OUT_LIDAR_SHP}")
print(f"📁 FABDEM 10 m:   {OUT_FABDEM_10M}")
print(f"📁 FABDEM 30 m:   {OUT_FABDEM_30M}")
print("📥 Source:         .npz only (no external 30 m dir)")
print("\nNaming:")
print("  {n}_LiDAR_{pXXXXX}.shp")
print("  {n}_FABDEM_10mbicubic_{pXXXXX}.tif")
print("  {n}_FABDEM_30m_{pXXXXX}.tif")


Found 70 tensor files in D:\Vaibhav\dem-lidar\temp
Mode: all products from .npz only (30 m = 3× block-mean of Ch1)

[1/70] 00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00070.npz | size 3000x1942 | photons 0 | EPSG:32644 | p00070
   ⚠️ no photons in mask — writing empty shapefile schema only


C:\Users\RRSCW\AppData\Local\Temp\ipykernel_12980\3115567510.py:60: FionaDeprecationWarning: This function will be removed in version 2.0. Please use CRS.from_epsg() instead.
  write_lidar_shapefile_full(


   ✅ FAB10  → 0_FABDEM_10mbicubic_p00070.tif  (3000x1942)
   ✅ FAB30  → 0_FABDEM_30m_p00070.tif  (1000x647 @ ~30.0 m) [block-mean ×3 from Ch1]

[2/70] 00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00071.npz | size 3000x1942 | photons 0 | EPSG:32644 | p00071
   ⚠️ no photons in mask — writing empty shapefile schema only
   ✅ FAB10  → 0_FABDEM_10mbicubic_p00071.tif  (3000x1942)
   ✅ FAB30  → 0_FABDEM_30m_p00071.tif  (1000x647 @ ~30.0 m) [block-mean ×3 from Ch1]

[3/70] 00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00072.npz | size 3000x1942 | photons 0 | EPSG:32644 | p00072
   ⚠️ no photons in mask — writing empty shapefile schema only
   ✅ FAB10  → 0_FABDEM_10mbicubic_p00072.tif  (3000x1942)
   ✅ FAB30  → 0_FABDEM_30m_p00072.tif  (1000x647 @ ~30.0 m) [block-mean ×3 from Ch1]

[4/70] 00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00073.npz | size 3000x1942 | photons 0 | EPSG:32644 |

C:\Users\RRSCW\AppData\Local\Temp\ipykernel_12980\3115567510.py:69: FionaDeprecationWarning: This function will be removed in version 2.0. Please use CRS.from_epsg() instead.
  write_lidar_shapefile_full(


   ✅ LiDAR  → 278_LiDAR_p00059.shp  (278 points)
   ✅ FAB10  → 278_FABDEM_10mbicubic_p00059.tif  (3000x1961)
   ✅ FAB30  → 278_FABDEM_30m_p00059.tif  (1000x653 @ ~30.0 m) [block-mean ×3 from Ch1]

[7/70] 00284_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00058.npz | size 3000x1961 | photons 284 | EPSG:32644 | p00058
   ✅ LiDAR  → 284_LiDAR_p00058.shp  (284 points)
   ✅ FAB10  → 284_FABDEM_10mbicubic_p00058.tif  (3000x1961)
   ✅ FAB30  → 284_FABDEM_30m_p00058.tif  (1000x653 @ ~30.0 m) [block-mean ×3 from Ch1]

[8/70] 00284_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00060.npz | size 3000x1961 | photons 284 | EPSG:32644 | p00060
   ✅ LiDAR  → 284_LiDAR_p00060.shp  (284 points)
   ✅ FAB10  → 284_FABDEM_10mbicubic_p00060.tif  (3000x1961)
   ✅ FAB30  → 284_FABDEM_30m_p00060.tif  (1000x653 @ ~30.0 m) [block-mean ×3 from Ch1]

[9/70] 00292_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00061.npz | size

## 5. Optional: quick sanity check on one export

Uncomment / edit a basename after the loop finishes.


In [ ]:
# Example: inspect one written 10 m FABDEM
# sample = sorted(os.listdir(OUT_FABDEM_10M))[0]
# path = os.path.join(OUT_FABDEM_10M, sample)
# with rasterio.open(path) as src:
#     print(sample)
#     print("  shape:", src.shape, "crs:", src.crs)
#     print("  transform:", src.transform)
#     print("  elev range:", float(src.read(1).min()), "→", float(src.read(1).max()))
print("Uncomment the block above to inspect a sample GeoTIFF.")
